
# Rolling-Horizon MPC on 127 Held-Out Rye Days
## Dynamic-feasible terminal-anchor formulation

Main setting:

\[
H_p=6\ \mathrm{h}.
\]

### Why this version exists

The previous fixed day-ahead SOC reference was numerically valid on the smoke-test day, but it can become **unreachable** after several rolling updates, especially for short horizons such as \(H_p=3\) h. That caused the development horizon-sensitivity run to terminate with an infeasible MILP.

This version removes that structural issue.

At each hour \(t\):

1. observe the current measured PV, wind, load, price, and SOC;
2. construct one updated forecast for the entire remaining day;
3. solve a **remaining-day feasibility/anchor plan** from the current SOC to the required final SOC \(=0.50\);
4. take the anchor SOC at \(t+H_p\) as the local MPC terminal target;
5. solve the \(H_p\)-hour MPC problem;
6. implement only the first-hour action and roll forward.

Thus,

\[
SOC_{t+H_p}^{MPC}=SOC_{t+H_p}^{anchor},
\]

where the anchor is recomputed from the **current state and current forecast**. Because the anchor itself is feasible, its first \(H_p\) hours provide a feasible path to the terminal target. This avoids the infeasibility produced by a stale fixed reference.

No future measured trajectory is supplied to the optimizer; future measured values are used only offline to construct the controlled forecast/realization pairs.

The online recovery metric remains

\[
RR=
\frac{\bar C_{DET}-\bar C_{MPC}}
{\bar C_{DET}-\bar C_{PF}}\times100\%.
\]


In [1]:

# Run only if required in the active environment.
%pip install -q pyomo highspy pandas numpy scipy matplotlib


Note: you may need to restart the kernel to use updated packages.


In [3]:

from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import pyomo.environ as pyo

ROOT = Path.cwd()

def find_file(filename):
    candidates = [
        ROOT / filename,
        ROOT / "data" / filename,
        ROOT / "config" / filename,
        Path("/mnt/data") / filename,
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not locate {filename}. Tried: {candidates}")

DEV_FILE = find_file("rye_development_304days.csv")
OOS_FILE = find_file("rye_oos_127days.csv")
BASELINE_FILE = find_file("final_oos_det_sto_pf_daily_CORRECTED.csv")

PARAM_FILE = None
for pname in ["study_parameters_frozen_v8.json", "study_parameters_frozen_v7.json", "study_parameters_frozen_v6.json", "study_parameters_frozen_v5.json"]:
    try:
        PARAM_FILE = find_file(pname)
        break
    except FileNotFoundError:
        pass
if PARAM_FILE is None:
    raise FileNotFoundError("Could not locate study parameter JSON.")

dev = pd.read_csv(DEV_FILE)
oos = pd.read_csv(OOS_FILE)
baseline = pd.read_csv(BASELINE_FILE)

with open(PARAM_FILE, "r", encoding="utf-8") as f:
    CFG = json.load(f)

print("Development days:", dev["date_utc"].nunique())
print("Held-out OOS days:", oos["date_utc"].nunique())
print("Corrected baseline rows:", len(baseline))
print("Parameter file:", PARAM_FILE)


Development days: 304
Held-out OOS days: 127
Corrected baseline rows: 1016
Parameter file: E:\AIUB Ref. Book & Class Materials\MASTERS\2ND SEMESTER\OPTIMIZATION OF POWER SYSTEM OPERATION\FINAL TERM\Simulation\Rye_Microgrid_MPC_OOS\config\study_parameters_frozen_v8.json


In [4]:

T = CFG["time"]["T_hours"]
DT = CFG["time"]["delta_t_h"]

PV_RATED = CFG["rye_physical_system"]["pv_rated_kwp"]
WIND_RATED = CFG["rye_physical_system"]["wind_rated_kw"]
BESS_E = CFG["rye_physical_system"]["bess_energy_kwh"]
P_CH_MAX = CFG["rye_physical_system"]["bess_charge_max_kw"]
P_DIS_MAX = CFG["rye_physical_system"]["bess_discharge_max_kw"]
ETA_CH = CFG["rye_physical_system"]["eta_charge"]
ETA_DIS = CFG["rye_physical_system"]["eta_discharge"]

SOC_MIN = CFG["bess_study_settings"]["soc_min"]
SOC_MAX = CFG["bess_study_settings"]["soc_max"]
SOC_INIT = CFG["bess_study_settings"]["soc_initial"]
SOC_TERM = CFG["bess_study_settings"]["soc_terminal"]

DG_MIN = CFG["study_added_diesel_generator"]["p_min_kw"]
DG_MAX = CFG["study_added_diesel_generator"]["p_max_kw"]
C_DG = CFG["study_added_diesel_generator"]["marginal_cost_nok_per_kwh"]

GRID_IMPORT_MAX = CFG["grid"]["import_max_kw"]
GRID_EXPORT_MAX = CFG["grid"]["export_max_kw"]
GRID_TARIFF = CFG["grid"]["energy_tariff_nok_per_kwh"]
SELL_FACTOR = CFG["grid"]["sell_price_fraction_of_buy"]

CRIT_SHARE = CFG["load_priority"]["critical_fraction"]
NCRIT_SHARE = CFG["load_priority"]["noncritical_fraction"]

C_BATT = CFG["objective_coefficients"]["battery_throughput_nok_per_kwh"]
C_CURT = CFG["objective_coefficients"]["renewable_curtailment_nok_per_kwh"]
C_SHED_N = CFG["objective_coefficients"]["noncritical_shedding_penalty_nok_per_kwh"]
C_SHED_C = CFG["objective_coefficients"]["critical_shedding_penalty_nok_per_kwh"]

ALPHAS = CFG["uncertainty"]["alpha_levels"]
BASE_SEED = CFG["uncertainty"]["random_seed"]
CONFIDENCE = CFG["evaluation"]["confidence_level"]

HP_MAIN = CFG.get("mpc", {}).get("main_prediction_horizon_h", 6)
HP_SENS = CFG.get("mpc", {}).get("supplementary_horizon_sensitivity_h", [3, 6, 12])

MPC_MIP_REL_GAP = 1e-6
MPC_MIP_ABS_GAP = 1e-7

assert len(baseline) == 127 * len(ALPHAS) * 2
assert HP_MAIN == 6

print("Main MPC horizon:", HP_MAIN, "h")
print("Supplementary horizons:", HP_SENS)


Main MPC horizon: 6 h
Supplementary horizons: [3, 6, 12]



## Rolling forecast model

The current hour is measured exactly:

\[
\sigma_0=0.
\]

For a future lead \(\ell\ge1\),

\[
\sigma_\ell=\alpha\sqrt{\frac{\ell}{24}}.
\]

The measured Rye trajectory remains hidden ground truth for future hours. It is used only offline to construct controlled forecast/realization pairs.


In [5]:

def get_day(df, date_string):
    x = df[df["date_utc"].astype(str) == str(date_string)].copy()
    x = x.sort_values("hour_utc")
    if len(x) != 24:
        raise ValueError(f"{date_string}: expected 24 rows, got {len(x)}")

    day = pd.DataFrame({
        "hour": np.arange(24),
        "pv": x["pv_clean_kw"].to_numpy(float),
        "wind": x["wind_clean_kw"].to_numpy(float),
        "load": x["load_kw"].to_numpy(float),
        "spot": x["spot_price_nok_per_kwh"].to_numpy(float),
    })
    day["buy_price"] = day["spot"] + GRID_TARIFF
    day["sell_price"] = SELL_FACTOR * day["buy_price"]
    return day


def oos_forecast_seed(date_string, alpha, base_seed=BASE_SEED):
    # Exact seed rule used in the frozen DET/STO baseline.
    date_num = int(str(date_string).replace("-", ""))
    alpha_num = int(round(float(alpha) * 1000))
    return (
        int(base_seed)
        + date_num * 1009
        + alpha_num * 9176
    ) % (2**32 - 1)


def make_day_ahead_forecast(actual_day, alpha, date_string):
    # Exact controlled point-forecast construction used by the DET baseline.
    rng = np.random.default_rng(oos_forecast_seed(date_string, alpha))
    fc = actual_day.copy()

    for col, cap in [("pv", PV_RATED), ("wind", WIND_RATED), ("load", None)]:
        realized = actual_day[col].to_numpy(float)
        e = rng.normal(0.0, float(alpha), size=24)
        e = np.clip(e, -0.80, 1.50)

        forecast = realized / (1.0 + e)
        forecast = np.maximum(forecast, 0.0)

        if cap is not None:
            forecast = np.minimum(forecast, cap)

        fc[col] = forecast

    return fc


def mpc_seed(date_string, alpha, current_hour, base_seed=BASE_SEED):
    # Reproducible rolling-update forecast seed.
    date_num = int(str(date_string).replace("-", ""))
    alpha_num = int(round(float(alpha) * 1000))
    return (
        int(base_seed)
        + date_num * 1009
        + alpha_num * 9176
        + int(current_hour) * 7919
    ) % (2**32 - 1)


def build_mpc_forecast_horizon(actual_day, date_string, alpha, current_hour, hp):
    """
    Current hour is observed exactly.
    Future hours use lead-dependent controlled forecast uncertainty.
    """
    H = min(int(hp), 24 - int(current_hour))
    rng = np.random.default_rng(mpc_seed(date_string, alpha, current_hour))

    out = actual_day.iloc[current_hour:current_hour + H].copy().reset_index(drop=True)

    for col, cap in [("pv", PV_RATED), ("wind", WIND_RATED), ("load", None)]:
        realized = out[col].to_numpy(float)
        forecast = realized.copy()

        # k=0 is the currently observed hour.
        for k in range(1, H):
            sigma = float(alpha) * np.sqrt(k / 24.0)
            e = rng.normal(0.0, sigma)
            e = np.clip(e, -0.80, 1.50)

            forecast[k] = realized[k] / (1.0 + e)
            forecast[k] = max(forecast[k], 0.0)

            if cap is not None:
                forecast[k] = min(forecast[k], cap)

        out[col] = forecast

    return out


_counts = oos.groupby("date_utc").size()
assert len(_counts) == 127
assert (_counts == 24).all()

print("Rolling forecast and OOS integrity checks PASSED.")


Rolling forecast and OOS integrity checks PASSED.


In [6]:

def configure_highs_solver(rel_gap=MPC_MIP_REL_GAP, abs_gap=MPC_MIP_ABS_GAP):
    for name in ("appsi_highs", "highs"):
        try:
            s = pyo.SolverFactory(name)
            if s is None or not s.available(exception_flag=False):
                continue

            if name == "appsi_highs":
                try:
                    s.config.mip_gap = rel_gap
                except Exception:
                    pass
                try:
                    s.highs_options["mip_rel_gap"] = rel_gap
                    s.highs_options["mip_abs_gap"] = abs_gap
                except Exception:
                    pass
            else:
                try:
                    s.options["mip_rel_gap"] = rel_gap
                    s.options["mip_abs_gap"] = abs_gap
                except Exception:
                    pass

            print(f"Using {name}: rel_gap={rel_gap:g}, abs_gap={abs_gap:g}")
            return s
        except Exception:
            pass

    raise RuntimeError("HiGHS not found. Run: pip install pyomo highspy")

solver = configure_highs_solver()


Using appsi_highs: rel_gap=1e-06, abs_gap=1e-07


In [7]:

def build_mpc_model(horizon_df, mode, current_soc, terminal_soc_target):
    if mode not in {"GC", "IS"}:
        raise ValueError("mode must be GC or IS")

    H = len(horizon_df)
    g = 1 if mode == "GC" else 0

    m = pyo.ConcreteModel(name=f"MPC_{mode}_H{H}")
    m.K = pyo.RangeSet(0, H - 1)
    m.Ksoc = pyo.RangeSet(0, H)

    m.pv_av = pyo.Param(
        m.K,
        initialize={k: float(horizon_df["pv"].iloc[k]) for k in range(H)},
        within=pyo.NonNegativeReals,
    )
    m.w_av = pyo.Param(
        m.K,
        initialize={k: float(horizon_df["wind"].iloc[k]) for k in range(H)},
        within=pyo.NonNegativeReals,
    )
    m.P_load = pyo.Param(
        m.K,
        initialize={k: float(horizon_df["load"].iloc[k]) for k in range(H)},
        within=pyo.NonNegativeReals,
    )
    m.cbuy = pyo.Param(
        m.K,
        initialize={k: float(horizon_df["buy_price"].iloc[k]) for k in range(H)},
    )
    m.csell = pyo.Param(
        m.K,
        initialize={k: float(horizon_df["sell_price"].iloc[k]) for k in range(H)},
    )

    m.u_dg = pyo.Var(m.K, domain=pyo.Binary)
    m.y_b = pyo.Var(m.K, domain=pyo.Binary)

    m.P_pv = pyo.Var(m.K, domain=pyo.NonNegativeReals)
    m.P_w = pyo.Var(m.K, domain=pyo.NonNegativeReals)
    m.P_dg = pyo.Var(m.K, domain=pyo.NonNegativeReals)
    m.P_ch = pyo.Var(m.K, domain=pyo.NonNegativeReals)
    m.P_dis = pyo.Var(m.K, domain=pyo.NonNegativeReals)
    m.SOC = pyo.Var(m.Ksoc, bounds=(SOC_MIN, SOC_MAX))
    m.P_buy = pyo.Var(m.K, domain=pyo.NonNegativeReals)
    m.P_sell = pyo.Var(m.K, domain=pyo.NonNegativeReals)
    m.shed_c = pyo.Var(m.K, domain=pyo.NonNegativeReals)
    m.shed_n = pyo.Var(m.K, domain=pyo.NonNegativeReals)
    m.curt = pyo.Var(m.K, domain=pyo.NonNegativeReals)

    m.cost = pyo.Objective(
        expr=sum(
            (
                m.cbuy[k] * m.P_buy[k]
                - m.csell[k] * m.P_sell[k]
                + C_DG * m.P_dg[k]
                + C_BATT * (m.P_ch[k] + m.P_dis[k])
                + C_CURT * m.curt[k]
                + C_SHED_C * m.shed_c[k]
                + C_SHED_N * m.shed_n[k]
            ) * DT
            for k in m.K
        ),
        sense=pyo.minimize,
    )

    m.pvlim = pyo.Constraint(m.K, rule=lambda m, k: m.P_pv[k] <= m.pv_av[k])
    m.wlim = pyo.Constraint(m.K, rule=lambda m, k: m.P_w[k] <= m.w_av[k])
    m.curtdef = pyo.Constraint(
        m.K,
        rule=lambda m, k:
            m.curt[k] == (m.pv_av[k] - m.P_pv[k]) + (m.w_av[k] - m.P_w[k]),
    )

    m.dgmin = pyo.Constraint(
        m.K, rule=lambda m, k: m.P_dg[k] >= DG_MIN * m.u_dg[k]
    )
    m.dgmax = pyo.Constraint(
        m.K, rule=lambda m, k: m.P_dg[k] <= DG_MAX * m.u_dg[k]
    )

    m.soc0 = pyo.Constraint(expr=m.SOC[0] == float(current_soc))
    m.socterm = pyo.Constraint(expr=m.SOC[H] == float(terminal_soc_target))
    m.socdyn = pyo.Constraint(
        m.K,
        rule=lambda m, k:
            m.SOC[k + 1]
            == m.SOC[k]
            + ETA_CH * m.P_ch[k] * DT / BESS_E
            - m.P_dis[k] * DT / (ETA_DIS * BESS_E),
    )

    m.chlim = pyo.Constraint(
        m.K, rule=lambda m, k: m.P_ch[k] <= P_CH_MAX * m.y_b[k]
    )
    m.dislim = pyo.Constraint(
        m.K, rule=lambda m, k: m.P_dis[k] <= P_DIS_MAX * (1 - m.y_b[k])
    )

    m.shedclim = pyo.Constraint(
        m.K, rule=lambda m, k: m.shed_c[k] <= CRIT_SHARE * m.P_load[k]
    )
    m.shednlim = pyo.Constraint(
        m.K, rule=lambda m, k: m.shed_n[k] <= NCRIT_SHARE * m.P_load[k]
    )

    m.buylim = pyo.Constraint(
        m.K, rule=lambda m, k: m.P_buy[k] <= g * GRID_IMPORT_MAX
    )
    m.selllim = pyo.Constraint(
        m.K, rule=lambda m, k: m.P_sell[k] <= g * GRID_EXPORT_MAX
    )

    m.balance = pyo.Constraint(
        m.K,
        rule=lambda m, k:
            m.P_pv[k] + m.P_w[k] + m.P_dg[k] + m.P_dis[k] + m.P_buy[k]
            == m.P_load[k] - m.shed_c[k] - m.shed_n[k]
            + m.P_ch[k] + m.P_sell[k],
    )

    return m


In [8]:

def solve_checked(model, tee=False):
    t0 = time.perf_counter()
    result = solver.solve(model, tee=tee)
    elapsed = time.perf_counter() - t0

    term = str(result.solver.termination_condition).lower()
    if "optimal" not in term:
        raise RuntimeError(f"Solver termination: {result.solver.termination_condition}")
    return elapsed


def first_action(model):
    k = 0
    return {
        "pv": float(pyo.value(model.P_pv[k])),
        "wind": float(pyo.value(model.P_w[k])),
        "dg": float(pyo.value(model.P_dg[k])),
        "charge": float(pyo.value(model.P_ch[k])),
        "discharge": float(pyo.value(model.P_dis[k])),
        "buy": float(pyo.value(model.P_buy[k])),
        "sell": float(pyo.value(model.P_sell[k])),
        "shed_crit": float(pyo.value(model.shed_c[k])),
        "shed_ncrit": float(pyo.value(model.shed_n[k])),
        "curtailment": float(pyo.value(model.curt[k])),
        "soc_start": float(pyo.value(model.SOC[0])),
        "soc_end": float(pyo.value(model.SOC[1])),
    }


def realized_first_hour_cost(action, actual_row):
    return (
        float(actual_row["buy_price"]) * action["buy"]
        - float(actual_row["sell_price"]) * action["sell"]
        + C_DG * action["dg"]
        + C_BATT * (action["charge"] + action["discharge"])
        + C_CURT * action["curtailment"]
        + C_SHED_C * action["shed_crit"]
        + C_SHED_N * action["shed_ncrit"]
    ) * DT


def action_balance_residual(action, actual_row):
    lhs = (
        action["pv"] + action["wind"] + action["dg"]
        + action["discharge"] + action["buy"]
    )
    rhs = (
        float(actual_row["load"])
        - action["shed_crit"] - action["shed_ncrit"]
        + action["charge"] + action["sell"]
    )
    return lhs - rhs


In [9]:

def build_dynamic_anchor(
    actual_day,
    date_string,
    alpha,
    mode,
    current_hour,
    current_soc,
):
    """
    Recompute a feasible remaining-day deterministic anchor at every MPC update.

    The forecast is built once for all remaining hours. The current hour is exact;
    future values use the controlled lead-dependent forecast errors.

    The anchor ends exactly at SOC_TERM. Its SOC at the end of the local Hp window
    is then used as the local MPC terminal target.
    """
    remaining_forecast = build_mpc_forecast_horizon(
        actual_day=actual_day,
        date_string=date_string,
        alpha=alpha,
        current_hour=current_hour,
        hp=24,  # automatically truncates to end of day
    )

    anchor_model = build_mpc_model(
        horizon_df=remaining_forecast,
        mode=mode,
        current_soc=current_soc,
        terminal_soc_target=SOC_TERM,
    )

    try:
        anchor_solve_time = solve_checked(anchor_model)
    except RuntimeError as exc:
        raise RuntimeError(
            f"Dynamic remaining-day anchor infeasible: "
            f"date={date_string}, alpha={alpha}, mode={mode}, "
            f"current_hour={current_hour}, current_soc={current_soc:.6f}"
        ) from exc

    H_rem = len(remaining_forecast)

    anchor_soc = np.array(
        [float(pyo.value(anchor_model.SOC[k])) for k in range(H_rem + 1)],
        dtype=float,
    )

    return (
        remaining_forecast,
        anchor_soc,
        float(anchor_solve_time),
        float(pyo.value(anchor_model.cost)),
    )


def full_day_pf_cost(actual_day, mode):
    """
    Development smoke-test benchmark only.
    """
    pf_model = build_mpc_model(
        horizon_df=actual_day,
        mode=mode,
        current_soc=SOC_INIT,
        terminal_soc_target=SOC_TERM,
    )
    _ = solve_checked(pf_model)
    return float(pyo.value(pf_model.cost))


In [10]:

def run_mpc_day(actual_day, date_string, alpha, mode, hp=HP_MAIN):
    current_soc = float(SOC_INIT)

    rows = []
    total_local_solve_time = 0.0
    total_anchor_solve_time = 0.0

    for t in range(24):
        # One consistent updated forecast for the entire remaining day.
        (
            remaining_forecast,
            anchor_soc,
            anchor_time,
            anchor_objective,
        ) = build_dynamic_anchor(
            actual_day=actual_day,
            date_string=date_string,
            alpha=alpha,
            mode=mode,
            current_hour=t,
            current_soc=current_soc,
        )

        total_anchor_solve_time += anchor_time

        H = min(int(hp), len(remaining_forecast))

        # IMPORTANT:
        # local horizon is sliced from the exact same forecast used by the anchor.
        horizon = remaining_forecast.iloc[:H].copy().reset_index(drop=True)

        # Guaranteed-feasible terminal target under this forecast:
        # the anchor's first H-hour trajectory is a feasible witness.
        terminal_target = float(anchor_soc[H])

        local_model = build_mpc_model(
            horizon_df=horizon,
            mode=mode,
            current_soc=current_soc,
            terminal_soc_target=terminal_target,
        )

        try:
            local_solve_time = solve_checked(local_model)
        except RuntimeError as exc:
            raise RuntimeError(
                f"Local MPC infeasible despite feasible anchor: "
                f"date={date_string}, alpha={alpha}, mode={mode}, "
                f"Hp={hp}, hour={t}, SOC={current_soc:.6f}, "
                f"terminal_target={terminal_target:.6f}"
            ) from exc

        total_local_solve_time += local_solve_time

        action = first_action(local_model)
        actual_row = actual_day.iloc[t]

        hour_cost = realized_first_hour_cost(action, actual_row)
        residual = action_balance_residual(action, actual_row)

        rows.append({
            "hour": t,
            "cost_nok": hour_cost,
            "critical_eens_kwh": action["shed_crit"] * DT,
            "noncritical_eens_kwh": action["shed_ncrit"] * DT,
            "diesel_kwh": action["dg"] * DT,
            "curtailment_kwh": action["curtailment"] * DT,
            "grid_import_kwh": action["buy"] * DT,
            "grid_export_kwh": action["sell"] * DT,
            "soc_start": action["soc_start"],
            "soc_end": action["soc_end"],
            "anchor_soc_window_end": terminal_target,
            "anchor_objective_nok": anchor_objective,
            "local_solve_time_s": local_solve_time,
            "anchor_solve_time_s": anchor_time,
            "balance_residual_kw": residual,
        })

        # Implement first action only.
        current_soc = action["soc_end"]

    hourly = pd.DataFrame(rows)

    metrics = {
        "mpc_cost_nok": float(hourly["cost_nok"].sum()),
        "mpc_critical_eens_kwh": float(hourly["critical_eens_kwh"].sum()),
        "mpc_noncritical_eens_kwh": float(hourly["noncritical_eens_kwh"].sum()),
        "mpc_critical_interrupted": int(
            hourly["critical_eens_kwh"].sum() > 1e-6
        ),
        "mpc_diesel_kwh": float(hourly["diesel_kwh"].sum()),
        "mpc_curtailment_kwh": float(hourly["curtailment_kwh"].sum()),
        "mpc_grid_import_kwh": float(hourly["grid_import_kwh"].sum()),
        "mpc_grid_export_kwh": float(hourly["grid_export_kwh"].sum()),

        "mpc_total_solve_time_s": float(
            total_local_solve_time + total_anchor_solve_time
        ),
        "mpc_total_local_solve_time_s": float(total_local_solve_time),
        "mpc_total_anchor_solve_time_s": float(total_anchor_solve_time),
        "mpc_mean_hourly_local_solve_time_s": float(
            hourly["local_solve_time_s"].mean()
        ),
        "mpc_mean_hourly_anchor_solve_time_s": float(
            hourly["anchor_solve_time_s"].mean()
        ),

        "mpc_final_soc": float(current_soc),
        "mpc_max_balance_residual_kw": float(
            hourly["balance_residual_kw"].abs().max()
        ),
    }

    return metrics, hourly



## Dynamic-anchor development smoke test

This test verifies:

\[
C_{PF}\le C_{MPC},
\]

exact final SOC,

\[
SOC_{24}=0.50,
\]

and numerical power balance.

The dynamic anchor is recomputed every hour, so this smoke test also verifies that the remaining-day anchor stays feasible throughout the entire 24-hour rolling simulation.


In [12]:

SMOKE_DATE = "2020-03-26"
SMOKE_ALPHA = 0.20

smoke_actual = get_day(dev, SMOKE_DATE)
smoke_rows = []

for mode in ["GC", "IS"]:
    pf_cost = full_day_pf_cost(smoke_actual, mode)

    met, hourly = run_mpc_day(
        smoke_actual,
        date_string=SMOKE_DATE,
        alpha=SMOKE_ALPHA,
        mode=mode,
        hp=HP_MAIN,
    )

    smoke_rows.append({
        "date": SMOKE_DATE,
        "mode": mode,
        "alpha": SMOKE_ALPHA,
        "pf_cost_nok": pf_cost,
        **met,
    })

smoke = pd.DataFrame(smoke_rows)
display(smoke)

assert np.allclose(smoke["mpc_final_soc"], SOC_TERM, atol=1e-6)
assert (smoke["mpc_max_balance_residual_kw"] <= 1e-6).all()

# PF should remain the lower-bound benchmark up to numerical tolerance.
assert (smoke["pf_cost_nok"] <= smoke["mpc_cost_nok"] + 1e-5).all()

print("MPC corrected development smoke test PASSED.")


,date,mode,alpha,pf_cost_nok,mpc_cost_nok,mpc_critical_eens_kwh,mpc_noncritical_eens_kwh,mpc_critical_interrupted,mpc_diesel_kwh,mpc_curtailment_kwh,mpc_grid_import_kwh,mpc_grid_export_kwh,mpc_total_solve_time_s,mpc_total_local_solve_time_s,mpc_total_anchor_solve_time_s,mpc_mean_hourly_local_solve_time_s,mpc_mean_hourly_anchor_solve_time_s,mpc_final_soc,mpc_max_balance_residual_kw
0,2020-03-26,GC,0.2,23.495123,23.495123,0.0,0.0,0,-1.221245e-15,8.881784e-16,216.151364,121.187755,1.719447,0.619398,1.100049,0.025808,0.045835,0.5,0.000000e+00
1,2020-03-26,IS,0.2,511.129606,512.153932,0.0,0.0,0,1.133767e+02,2.349372e-01,0.000000,0.000000,2.218149,0.845869,1.372280,0.035245,0.057178,0.5,3.552714e-15


MPC corrected development smoke test PASSED.


In [15]:
for fname in [
    "mpc_horizon_sensitivity_checkpoint.csv",
    "mpc_horizon_sensitivity_development.csv",
]:
    p = ROOT / fname
    if p.exists():
        p.unlink()
        print("Deleted old file:", p)

In [16]:
print("ROOT =", ROOT)

for p in ROOT.glob("mpc_horizon_sensitivity*"):
    print(p.name)

ROOT = E:\AIUB Ref. Book & Class Materials\MASTERS\2ND SEMESTER\OPTIMIZATION OF POWER SYSTEM OPERATION\FINAL TERM\Simulation\Rye_Microgrid_MPC_OOS



## Optional supplementary \(H_p=\{3,6,12\}\) sensitivity on development-only days


In [18]:

def evenly_spaced_development_days(dev_df, n_days=24):
    dates = np.array(sorted(dev_df["date_utc"].astype(str).unique()))
    idx = np.linspace(0, len(dates) - 1, n_days).round().astype(int)
    return list(dates[idx])

DEV_SENS_DAYS = evenly_spaced_development_days(dev, 24)

HP_SENS_CHECKPOINT = ROOT / "mpc_horizon_sensitivity_checkpoint.csv"
HP_SENS_FILE = ROOT / "mpc_horizon_sensitivity_development.csv"


def run_horizon_sensitivity(
    dev_df,
    days=DEV_SENS_DAYS,
    alpha=0.20,
    horizons=(3, 6, 12),
    modes=("GC", "IS"),
):
    """
    Development-only Hp sensitivity with checkpoint/resume.

    A row is saved after every completed (date, mode, Hp) case.
    """
    if HP_SENS_CHECKPOINT.exists():
        out = pd.read_csv(HP_SENS_CHECKPOINT)
        done = set(zip(
            out["date"].astype(str),
            out["mode"].astype(str),
            out["Hp"].astype(int),
        ))
        print(
            f"Resuming horizon-sensitivity checkpoint "
            f"with {len(out)} completed cases."
        )
    else:
        out = pd.DataFrame()
        done = set()

    total_cases = len(days) * len(modes) * len(horizons)

    for i, date in enumerate(days, start=1):
        actual = get_day(dev_df, date)

        for mode in modes:
            for hp in horizons:
                key = (str(date), str(mode), int(hp))
                if key in done:
                    continue

                try:
                    met, _ = run_mpc_day(
                        actual_day=actual,
                        date_string=date,
                        alpha=alpha,
                        mode=mode,
                        hp=hp,
                    )
                except Exception as exc:
                    print(
                        f"FAILED case: date={date}, mode={mode}, Hp={hp}, "
                        f"alpha={alpha}"
                    )
                    raise

                row = pd.DataFrame([{
                    "date": date,
                    "mode": mode,
                    "alpha": float(alpha),
                    "Hp": int(hp),
                    **met,
                }])

                out = pd.concat([out, row], ignore_index=True)
                out = out.sort_values(
                    ["date", "mode", "Hp"]
                ).reset_index(drop=True)
                out.to_csv(HP_SENS_CHECKPOINT, index=False)

                done.add(key)

        print(
            f"Horizon sensitivity completed {i}/{len(days)} development days "
            f"({len(out)}/{total_cases} cases saved)"
        )

    out.to_csv(HP_SENS_FILE, index=False)
    print("Development Hp sensitivity saved:", HP_SENS_FILE)
    return out

hp_sens = run_horizon_sensitivity(
    dev,
    alpha=0.20,
    horizons=[3, 6, 12],
    modes=("GC", "IS"),
)
#
# hp_summary = (
#     hp_sens
#     .groupby(["mode", "Hp"])
#     .agg(
#         mean_cost_nok=("mpc_cost_nok", "mean"),
#         median_cost_nok=("mpc_cost_nok", "median"),
#         mean_critical_eens_kwh=("mpc_critical_eens_kwh", "mean"),
#         mean_noncritical_eens_kwh=("mpc_noncritical_eens_kwh", "mean"),
#         mean_total_solve_time_s=("mpc_total_solve_time_s", "mean"),
#         mean_final_soc=("mpc_final_soc", "mean"),
#         max_balance_residual_kw=("mpc_max_balance_residual_kw", "max"),
#     )
#     .reset_index()
# )
#
# display(hp_summary)

Horizon sensitivity completed 1/24 development days (6/144 cases saved)
Horizon sensitivity completed 2/24 development days (12/144 cases saved)
Horizon sensitivity completed 3/24 development days (18/144 cases saved)
Horizon sensitivity completed 4/24 development days (24/144 cases saved)
Horizon sensitivity completed 5/24 development days (30/144 cases saved)
Horizon sensitivity completed 6/24 development days (36/144 cases saved)
Horizon sensitivity completed 7/24 development days (42/144 cases saved)
Horizon sensitivity completed 8/24 development days (48/144 cases saved)
Horizon sensitivity completed 9/24 development days (54/144 cases saved)
Horizon sensitivity completed 10/24 development days (60/144 cases saved)
Horizon sensitivity completed 11/24 development days (66/144 cases saved)
Horizon sensitivity completed 12/24 development days (72/144 cases saved)
Horizon sensitivity completed 13/24 development days (78/144 cases saved)
FAILED case: date=2020-06-21, mode=IS, Hp=3, alp

RuntimeError: Dynamic remaining-day anchor infeasible: date=2020-06-21, alpha=0.2, mode=IS, current_hour=19, current_soc=0.668481


## Final 127-day MPC OOS evaluation

Run this only after the dynamic-anchor smoke test and development-only horizon sensitivity are accepted.

There are

\[
127\times4\times2\times24=24{,}384
\]

implemented rolling MPC updates. Each update solves:

1. one remaining-day anchor MILP, and
2. one local \(H_p\)-hour MPC MILP.

Therefore the final run contains approximately

\[
48{,}768
\]

MILP solves. Checkpoint/resume remains enabled at the daily-case level.


In [ ]:

MPC_CHECKPOINT = ROOT / "mpc_oos_checkpoint.csv"
MPC_DAILY_FILE = ROOT / "mpc_oos_daily.csv"


def completed_mpc_keys(df):
    if df is None or len(df) == 0:
        return set()
    return set(zip(
        df["date"].astype(str),
        df["alpha"].astype(float),
        df["mode"].astype(str),
    ))


def run_final_mpc_oos(oos_df, alphas=ALPHAS, modes=("GC", "IS"), hp=HP_MAIN):
    dates = sorted(oos_df["date_utc"].astype(str).unique())

    if MPC_CHECKPOINT.exists():
        out = pd.read_csv(MPC_CHECKPOINT)
        done = completed_mpc_keys(out)
        print(f"Resuming checkpoint with {len(out)} completed daily cases.")
    else:
        out = pd.DataFrame()
        done = set()

    new_rows = []

    for day_i, date in enumerate(dates, start=1):
        actual = get_day(oos_df, date)

        for alpha in alphas:
            for mode in modes:
                key = (date, float(alpha), mode)
                if key in done:
                    continue

                met, _ = run_mpc_day(actual, date, alpha, mode, hp=hp)
                new_rows.append({
                    "date": date,
                    "mode": mode,
                    "alpha": float(alpha),
                    "Hp": int(hp),
                    **met,
                })
                done.add(key)

        if new_rows:
            add = pd.DataFrame(new_rows)
            out = pd.concat([out, add], ignore_index=True)
            new_rows = []
            out = out.sort_values(["date", "alpha", "mode"]).reset_index(drop=True)
            out.to_csv(MPC_CHECKPOINT, index=False)

        print(
            f"Completed {day_i}/{len(dates)} OOS days "
            f"({len(out)}/{len(dates)*len(alphas)*len(modes)} daily MPC cases saved)"
        )

    out.to_csv(MPC_DAILY_FILE, index=False)
    print("Final MPC daily file saved:", MPC_DAILY_FILE)
    return out


# FINAL RUN — only after smoke test passes:
# mpc_daily = run_final_mpc_oos(oos)
# display(mpc_daily.head())


In [ ]:

def merge_mpc_with_baseline(mpc_daily, baseline_df):
    merged = baseline_df.merge(
        mpc_daily,
        on=["date", "mode", "alpha"],
        how="inner",
        validate="one_to_one",
    )

    expected = 127 * len(ALPHAS) * 2
    assert len(merged) == expected, f"Expected {expected} rows, found {len(merged)}"

    merged["mpc_savings_vs_det_nok"] = merged["det_cost_nok"] - merged["mpc_cost_nok"]
    merged["mpc_residual_gap_vs_pf_nok"] = merged["mpc_cost_nok"] - merged["pf_cost_nok"]
    return merged


# After final MPC run:
# all_daily = merge_mpc_with_baseline(mpc_daily, baseline)
# all_daily.to_csv("final_oos_det_sto_mpc_pf_daily.csv", index=False)
# display(all_daily.head())


In [ ]:

def validate_mpc_results(all_daily, tol=1e-5):
    pf_minus_mpc = all_daily["pf_cost_nok"] - all_daily["mpc_cost_nok"]

    violations = int((pf_minus_mpc > tol).sum())
    max_pf_minus_mpc = float(pf_minus_mpc.max())
    max_soc_error = float(np.max(np.abs(all_daily["mpc_final_soc"].to_numpy(float) - SOC_TERM)))
    max_balance_residual = float(all_daily["mpc_max_balance_residual_kw"].max())

    print("PF > MPC violations:", violations)
    print("Maximum PF - MPC =", max_pf_minus_mpc)
    print("Maximum final SOC error =", max_soc_error)
    print("Maximum hourly balance residual =", max_balance_residual)

    assert violations == 0
    assert max_soc_error <= 1e-6
    assert max_balance_residual <= 1e-6

    print("Final MPC numerical validation PASSED.")


# After merge:
# validate_mpc_results(all_daily)



## Final RR, paired 95% confidence intervals, and four-strategy summary


In [ ]:

def paired_mean_ci(values, confidence=CONFIDENCE):
    x = np.asarray(values, dtype=float)
    n = len(x)
    mean = float(x.mean())

    if n < 2:
        return mean, np.nan, np.nan

    se = stats.sem(x)
    if not np.isfinite(se) or se == 0:
        return mean, mean, mean

    h = stats.t.ppf((1 + confidence) / 2, n - 1) * se
    return mean, mean - h, mean + h


def summarize_all_strategies(all_daily):
    rows = []

    for (mode, alpha), g in all_daily.groupby(["mode", "alpha"]):
        det = g["det_cost_nok"].to_numpy(float)
        sto = g["sto_cost_nok"].to_numpy(float)
        mpc = g["mpc_cost_nok"].to_numpy(float)
        pf = g["pf_cost_nok"].to_numpy(float)

        cou_daily = det - pf
        sto_save_daily = det - sto
        mpc_save_daily = det - mpc
        mpc_residual_daily = mpc - pf

        cou, cou_lo, cou_hi = paired_mean_ci(cou_daily)
        sto_save, sto_lo, sto_hi = paired_mean_ci(sto_save_daily)
        mpc_save, mpc_lo, mpc_hi = paired_mean_ci(mpc_save_daily)
        mpc_res, mpc_res_lo, mpc_res_hi = paired_mean_ci(mpc_residual_daily)

        det_mean = det.mean()
        sto_mean = sto.mean()
        mpc_mean = mpc.mean()
        pf_mean = pf.mean()

        smr = 100.0 * sto_save / cou if abs(cou) > 1e-12 else np.nan
        rr = 100.0 * mpc_save / cou if abs(cou) > 1e-12 else np.nan

        rows.append({
            "mode": mode,
            "alpha": alpha,

            "det_mean_cost_nok": det_mean,
            "sto_mean_cost_nok": sto_mean,
            "mpc_mean_cost_nok": mpc_mean,
            "pf_mean_cost_nok": pf_mean,

            "cou_nok": cou,
            "cou_ci95_low": cou_lo,
            "cou_ci95_high": cou_hi,

            "sto_savings_vs_det_nok": sto_save,
            "sto_savings_ci95_low": sto_lo,
            "sto_savings_ci95_high": sto_hi,
            "smr_percent": smr,

            "mpc_savings_vs_det_nok": mpc_save,
            "mpc_savings_ci95_low": mpc_lo,
            "mpc_savings_ci95_high": mpc_hi,
            "rr_percent": rr,

            "mpc_residual_gap_vs_pf_nok": mpc_res,
            "mpc_residual_ci95_low": mpc_res_lo,
            "mpc_residual_ci95_high": mpc_res_hi,

            "det_mean_critical_eens_kwh": g["det_critical_eens_kwh"].mean(),
            "sto_mean_critical_eens_kwh": g["sto_critical_eens_kwh"].mean(),
            "mpc_mean_critical_eens_kwh": g["mpc_critical_eens_kwh"].mean(),

            "det_mean_noncritical_eens_kwh": g["det_noncritical_eens_kwh"].mean(),
            "sto_mean_noncritical_eens_kwh": g["sto_noncritical_eens_kwh"].mean(),
            "mpc_mean_noncritical_eens_kwh": g["mpc_noncritical_eens_kwh"].mean(),

            "det_critical_interruption_probability": g["det_critical_interrupted"].mean(),
            "sto_critical_interruption_probability": g["sto_critical_interrupted"].mean(),
            "mpc_critical_interruption_probability": g["mpc_critical_interrupted"].mean(),

            "mean_mpc_total_solve_time_s": g["mpc_total_solve_time_s"].mean(),
            "mean_mpc_local_solve_time_s": g["mpc_total_local_solve_time_s"].mean(),
            "mean_mpc_anchor_solve_time_s": g["mpc_total_anchor_solve_time_s"].mean(),
        })

    return pd.DataFrame(rows).sort_values(["mode", "alpha"]).reset_index(drop=True)


# After validation:
# final_four_summary = summarize_all_strategies(all_daily)
# final_four_summary.to_csv("final_oos_det_sto_mpc_pf_summary.csv", index=False)
# display(final_four_summary)


In [ ]:

def plot_four_strategy_results(summary):
    for mode in ["GC", "IS"]:
        g = summary[summary["mode"] == mode].sort_values("alpha")

        fig, ax = plt.subplots(figsize=(6.8, 4.2))
        ax.plot(g["alpha"] * 100, g["det_mean_cost_nok"], marker="o", label="DET")
        ax.plot(g["alpha"] * 100, g["sto_mean_cost_nok"], marker="o", label="STO")
        ax.plot(g["alpha"] * 100, g["mpc_mean_cost_nok"], marker="o", label="MPC")
        ax.plot(g["alpha"] * 100, g["pf_mean_cost_nok"], marker="o", label="PF")
        ax.set_xlabel("Forecast-error standard deviation, alpha (%)")
        ax.set_ylabel("Mean realized operating cost (NOK/day)")
        ax.set_title(f"{mode}: cost comparison")
        ax.grid(alpha=0.25)
        ax.legend()
        plt.show()

    fig, ax = plt.subplots(figsize=(6.8, 4.2))
    for mode, g in summary.groupby("mode"):
        g = g.sort_values("alpha")
        ax.plot(g["alpha"] * 100, g["smr_percent"], marker="o", label=f"SMR-{mode}")
        ax.plot(g["alpha"] * 100, g["rr_percent"], marker="s", label=f"RR-{mode}")

    ax.set_xlabel("Forecast-error standard deviation, alpha (%)")
    ax.set_ylabel("Mitigated / recovered CoU (%)")
    ax.set_title("Ex-ante STO mitigation and online MPC recovery")
    ax.grid(alpha=0.25)
    ax.legend()
    plt.show()


# Optional:
# plot_four_strategy_results(final_four_summary)



After this notebook passes, the main information-value pipeline is complete:

\[
\text{Forecast uncertainty}
\rightarrow CoU
\rightarrow
\{SMR\text{ (ex-ante)}, RR\text{ (online)}\}
\rightarrow GC/IS
\rightarrow \text{reliability}.
\]


In [1]:
print(Hp)

NameError: name 'Hp' is not defined